In [ ]:


import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

dates = pd.date_range(start='2019-05-01', end='2025-05-22', freq='B')
n_days = len(dates)


price = 300.0
prices = []
for i in range(n_days):
    price *= (1 + np.random.normal(0.0006, 0.011))
    prices.append(price)

df = pd.DataFrame({
    'Date': dates,
    'Open': prices,
    'High': np.array(prices) * (1 + np.random.uniform(0.003, 0.018, n_days)),
    'Low': np.array(prices) * (1 - np.random.uniform(0.003, 0.018, n_days)),
    'Close': prices,
    'Volume': np.random.randint(55_000_000, 115_000_000, n_days)
}).set_index('Date')


df['High'] = df[['Open', 'Close']].max(axis=1) * 1.015
df['Low'] = df[['Open', 'Close']].min(axis=1) * 0.985

print(f"Generated {len(df)} days of data")

print("🔧 Performing Feature Engineering...")


df['Returns'] = df['Close'].pct_change()
df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
df['Price_Gap'] = (df['Open'] - df['Close'].shift(1)) / df['Close'].shift(1)
df['High_Low_Range'] = (df['High'] - df['Low']) / df['Close']
df['Close_Open_Ratio'] = df['Close'] / df['Open']


df['SMA10'] = df['Close'].rolling(10).mean()
df['SMA20'] = df['Close'].rolling(20).mean()
df['SMA50'] = df['Close'].rolling(50).mean()
df['SMA200'] = df['Close'].rolling(200).mean()

df['EMA9']  = df['Close'].ewm(span=9, adjust=False).mean()
df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
df['EMA50'] = df['Close'].ewm(span=50, adjust=False).mean()


df['MACD'] = df['EMA12'] - df['EMA26']
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']


delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = -delta.where(delta < 0, 0).rolling(14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))


df['BB_Middle'] = df['Close'].rolling(20).mean()
df['BB_Std'] = df['Close'].rolling(20).std()
df['BB_Upper'] = df['BB_Middle'] + 2 * df['BB_Std']
df['BB_Lower'] = df['BB_Middle'] - 2 * df['BB_Std']
df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / df['BB_Middle']


df['Volatility_14'] = df['Returns'].rolling(14).std()
df['Volatility_30'] = df['Returns'].rolling(30).std()
df['Momentum_5'] = df['Close'] - df['Close'].shift(5)
df['Momentum_10'] = df['Close'] - df['Close'].shift(10)
df['Momentum_20'] = df['Close'] - df['Close'].shift(20)


df['Volume_MA_10'] = df['Volume'].rolling(10).mean()
df['Volume_MA_20'] = df['Volume'].rolling(20).mean()
df['Volume_Ratio_10'] = df['Volume'] / df['Volume_MA_10']
df['Volume_Ratio_20'] = df['Volume'] / df['Volume_MA_20']
df['Volume_Change'] = df['Volume'].pct_change()


df['ROC_10'] = df['Close'].pct_change(10) * 100
df['Stoch_K'] = 100 * (df['Close'] - df['Low'].rolling(14).min()) / \
                (df['High'].rolling(14).max() - df['Low'].rolling(14).min() + 1e-8)

df = df.dropna()


df['Target'] = (df['Returns'].shift(-1) > 0).astype(int)
df = df.dropna()

print(f"✅ Feature Engineering Completed - Total Features: {df.shape[1]-2}")


print("⚙️ Preprocessing data...")

feature_cols = [col for col in df.columns if col not in ['Target']]


scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print("✅ Features Normalized")


def create_sequences(data, target, lookback=30):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data.iloc[i:i+lookback].values)
        y.append(target.iloc[i + lookback])
    return np.array(X), np.array(y)

lookback = 30
X_seq, y_seq = create_sequences(df[feature_cols], df['Target'], lookback)

print(f"✅ Sequences created: X = {X_seq.shape}, y = {y_seq.shape}")


train_size = int(0.70 * len(X_seq))
val_size   = int(0.15 * len(X_seq))

X_train = X_seq[:train_size]
X_val   = X_seq[train_size:train_size + val_size]
X_test  = X_seq[train_size + val_size:]

y_train = y_seq[:train_size]
y_val   = y_seq[train_size:train_size + val_size]
y_test  = y_seq[train_size + val_size:]

print("\n🎯 Final Split:")
print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")

df.to_excel("SPY_Phase1_Feature_Rich_Dataset.xlsx", index=True)
np.savez("SPY_Phase1_Sequences.npz",
         X_train=X_train, X_val=X_val, X_test=X_test,
         y_train=y_train, y_val=y_val, y_test=y_test)

print("\n✅ All files saved successfully!")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')


data = np.load("SPY_Phase1_Sequences.npz")
X_train = data['X_train']
X_val = data['X_val']
X_test = data['X_test']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']

print(f"Data Loaded → Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 64
device = torch.device('cpu')

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.25):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

def train_model(model, train_loader, val_loader, epochs=50, lr=0.0005, name="LSTM", y_val_ref=None):
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

    for epoch in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            model.eval()
            val_preds = []
            with torch.no_grad():
                for Xv, _ in val_loader:
                    val_preds.extend((model(Xv.to(device)).cpu().numpy() > 0.5).astype(int))
            val_f1 = f1_score(y_val_ref, val_preds, zero_division=0)
            scheduler.step(val_f1)
            print(f"{name} - Epoch {epoch+1:2d} | Val F1: {val_f1:.4f}")
    return model

print("\n⌐ Training Full-Feature LSTM...")
full_train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
full_val_loader = DataLoader(StockDataset(X_val, y_val), batch_size=batch_size)

full_model = LSTMModel(input_size=X_train.shape[2], hidden_size=128, num_layers=2, dropout=0.25)
full_model = train_model(full_model, full_train_loader, full_val_loader, name="Full LSTM", y_val_ref=y_val)

print("\n⌐ Training PCA-Reduced LSTM...")

# Flatten sequences for PCA
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat   = X_val.reshape(X_val.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

pca = PCA(n_components=0.93)
X_train_pca = pca.fit_transform(X_train_flat)
X_val_pca   = pca.transform(X_val_flat)
X_test_pca  = pca.transform(X_test_flat)

# Reshape back to (Samples, 1, Features) for LSTM
X_train_pca = X_train_pca.reshape(X_train_pca.shape[0], 1, X_train_pca.shape[1])
X_val_pca   = X_val_pca.reshape(X_val_pca.shape[0], 1, X_val_pca.shape[1])
X_test_pca  = X_test_pca.reshape(X_test_pca.shape[0], 1, X_test_pca.shape[1])

pca_train_loader = DataLoader(StockDataset(X_train_pca, y_train), batch_size=batch_size, shuffle=True)
pca_val_loader   = DataLoader(StockDataset(X_val_pca, y_val), batch_size=batch_size)

pca_model = LSTMModel(input_size=X_train_pca.shape[2], hidden_size=128, num_layers=2, dropout=0.25)
pca_model = train_model(pca_model, pca_train_loader, pca_val_loader, name="PCA LSTM", y_val_ref=y_val)

def evaluate(model, X, y_true, name):
    model.eval()
    loader = DataLoader(StockDataset(X, y_true), batch_size=128, shuffle=False)
    preds = []
    with torch.no_grad():
        for Xb, _ in loader:
            preds.extend((model(Xb.to(device)).cpu().numpy() > 0.5).astype(int).flatten())
    preds = np.array(preds)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, preds),
        'F1_Score': f1_score(y_true, preds, zero_division=0),
        'Precision': precision_score(y_true, preds, zero_division=0),
        'Recall': recall_score(y_true, preds, zero_division=0)
    }

print("\n" + "="*70)
print("██ PHASE 2 - LSTM BASELINES (FINAL)")
print("="*70)

results = [
    evaluate(full_model, X_test, y_test, "Full-Feature LSTM"),
    evaluate(pca_model, X_test_pca, y_test, "PCA-Reduced LSTM")
]

results_df = pd.DataFrame(results)
print(results_df.round(4))

results_df.to_excel("Phase2_LSTM_Benchmark_Final.xlsx", index=False)
print("\n✅ Results saved to 'Phase2_LSTM_Benchmark_Final.xlsx'")

In [ ]:
import numpy as np
import pandas as pd
import torch
import pennylane as qml
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("🚀 Starting Phase 3: Quantum Feature Mapping")

data = np.load("SPY_Phase1_Sequences.npz")
X_train = data['X_train']
X_val   = data['X_val']
X_test  = data['X_test']
y_train = data['y_train']
y_val   = data['y_val']
y_test  = data['y_test']

print(f"Original Data Shape: {X_train.shape}")

n_qubits = 8                    # Number of qubits (adjust based on performance)
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_feature_map(x):
    """Quantum Feature Map: Angle Embedding + Entanglement"""
    # Angle Embedding
    for i in range(n_qubits):
        qml.RY(x[i % len(x)], wires=i)      # Encode features into rotation

    # Entangling Layers (creates quantum correlations)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i+1])
    qml.CNOT(wires=[n_qubits-1, 0])         # Ring entanglement

    # Second layer of rotations
    for i in range(n_qubits):
        qml.RY(x[(i+3) % len(x)], wires=i)

    # Measure expectation values
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def generate_quantum_features(X, batch_size=32):
    X_quantum = []
    scaler = StandardScaler()

    # Use last timestep features for quantum mapping (most recent information)
    X_last = X[:, -1, :]                    # Shape: (samples, features)
    X_scaled = scaler.fit_transform(X_last)

    print(f"Generating Quantum Features for {len(X)} samples...")

    for i in tqdm(range(0, len(X_scaled), batch_size)):
        batch = X_scaled[i:i+batch_size]
        batch_quantum = []

        for sample in batch:
            # Use first n_qubits features
            feats = sample[:n_qubits]
            # Pad if needed
            if len(feats) < n_qubits:
                feats = np.pad(feats, (0, n_qubits - len(feats)))
            q_features = quantum_feature_map(feats)
            batch_quantum.append(q_features)

        X_quantum.extend(batch_quantum)

    return np.array(X_quantum)

print("Generating Quantum Features for Train, Val, Test...")

X_train_q = generate_quantum_features(X_train)
X_val_q   = generate_quantum_features(X_val)
X_test_q  = generate_quantum_features(X_test)

print(f"Quantum Features Generated → Shape: {X_train_q.shape}")

# Option 1: Use only Quantum Features
# X_train_hybrid = X_train_q
# X_test_hybrid = X_test_q

# Option 2: Concatenate Last timestep classical + Quantum (Recommended)
X_train_last = X_train[:, -1, :15]           # Take top 15 classical features
X_test_last  = X_test[:, -1, :15]

X_train_hybrid = np.concatenate([X_train_last, X_train_q], axis=1)
X_test_hybrid  = np.concatenate([X_test_last, X_test_q], axis=1)

print(f"Hybrid Feature Shape: {X_train_hybrid.shape}")

np.savez("SPY_Phase3_Quantum_Features.npz",
         X_train_hybrid=X_train_hybrid,
         X_val_hybrid=X_val_q,           # You can expand later
         X_test_hybrid=X_test_hybrid,
         y_train=y_train,
         y_val=y_val,
         y_test=y_test)

print("\n✅ Phase 3 Completed Successfully!")
print("Quantum-Enhanced dataset saved as 'SPY_Phase3_Quantum_Features.npz'")
print(f"New feature dimension: {X_train_hybrid.shape[1]}")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("🚀 Starting Phase 4: Hybrid Quantum-Classical Model")

data = np.load("SPY_Phase3_Quantum_Features.npz")
X_train_hybrid = data['X_train_hybrid']
X_test_hybrid  = data['X_test_hybrid']
y_train = data['y_train']
y_test  = data['y_test']

print(f"Hybrid Data Loaded - Train: {X_train_hybrid.shape} | Test: {X_test_hybrid.shape}")

class HybridStockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # Add timestep dimension for LSTM
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 64
train_dataset = HybridStockDataset(X_train_hybrid, y_train)
test_dataset  = HybridStockDataset(X_test_hybrid, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

device = torch.device('cpu')

class HybridQuantumLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.25):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])        # Use last timestep output
        return out

def train_hybrid_model(model, train_loader, epochs=40, lr=0.0008):
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.001)

    print("Training Hybrid Quantum-LSTM Model...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}')

    return model

input_size = X_train_hybrid.shape[1]

hybrid_model = HybridQuantumLSTM(input_size=input_size,
                                hidden_size=128,
                                num_layers=2,
                                dropout=0.25)

hybrid_model = train_hybrid_model(hybrid_model, train_loader, epochs=40)

def evaluate_hybrid(model, loader, name="Hybrid Model"):
    model.eval()
    y_pred, y_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            y_pred.extend((outputs.cpu().numpy() > 0.5).astype(int).flatten())
            y_true.extend(y_batch.numpy().flatten())

    y_pred = np.array(y_pred)
    y_true = np.array(y_true)

    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1_Score': f1_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0)
    }
    return metrics

# Final Results
print("\n" + "="*60)
print("📊 PHASE 4: HYBRID QUANTUM-LSTM RESULTS")
print("="*60)

hybrid_metrics = evaluate_hybrid(hybrid_model, test_loader, "Hybrid Quantum-LSTM")
results_df = pd.DataFrame([hybrid_metrics])
print(results_df.round(4))

# Save model and results
torch.save(hybrid_model.state_dict(), "hybrid_quantum_lstm_model.pth")
results_df.to_excel("Phase4_Hybrid_Results.xlsx", index=False)

print("\n✅ Phase 4 Completed!")
print("Model saved: 'hybrid_quantum_lstm_model.pth'")
print("Results saved: 'Phase4_Hybrid_Results.xlsx'")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print("🚀 Starting Phase 5: Comparative Evaluation")


# Load Phase 1 Data
data_phase1 = np.load("SPY_Phase1_Sequences.npz")
X_test = data_phase1['X_test']
y_test = data_phase1['y_test']

# Load Phase 3 Quantum Data
data_phase3 = np.load("SPY_Phase3_Quantum_Features.npz")
X_test_hybrid = data_phase3['X_test_hybrid']


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.25):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


full_model = LSTMModel(X_test.shape[2])
full_model.load_state_dict(torch.load("full_lstm_model.pth", map_location='cpu'))  # Adjust path if needed

pca_model = LSTMModel(45)  # Adjust input size based on your PCA
pca_model.load_state_dict(torch.load("pca_lstm_model.pth", map_location='cpu'))

hybrid_model = LSTMModel(X_test_hybrid.shape[1])
hybrid_model.load_state_dict(torch.load("hybrid_quantum_lstm_model.pth", map_location='cpu'))

device = torch.device('cpu')
full_model.eval()
pca_model.eval()
hybrid_model.eval()

def get_predictions(model, X):
    model.eval()
    X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(1) if len(X.shape) == 2 else torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
        outputs = model(X_tensor.to(device))
        preds = (outputs.cpu().numpy() > 0.5).astype(int).flatten()
    return preds

# Get predictions
full_pred = get_predictions(full_model, X_test)
# For PCA - reshape if needed
X_test_flat = X_test.reshape(X_test.shape[0], -1)
pca = PCA(n_components=0.93)
X_test_pca = pca.fit_transform(X_test_flat).reshape(-1, 1, -1)  # Simplified
pca_pred = get_predictions(pca_model, X_test_pca)

hybrid_pred = get_predictions(hybrid_model, X_test_hybrid)


results = {
    'Model': ['Full Classical LSTM', 'PCA LSTM', 'Hybrid Quantum-LSTM'],
    'Accuracy': [
        accuracy_score(y_test, full_pred),
        accuracy_score(y_test, pca_pred),
        accuracy_score(y_test, hybrid_pred)
    ],
    'F1_Score': [
        f1_score(y_test, full_pred, zero_division=0),
        f1_score(y_test, pca_pred, zero_division=0),
        f1_score(y_test, hybrid_pred, zero_division=0)
    ],
    'Precision': [
        precision_score(y_test, full_pred, zero_division=0),
        precision_score(y_test, pca_pred, zero_division=0),
        precision_score(y_test, hybrid_pred, zero_division=0)
    ],
    'Recall': [
        recall_score(y_test, full_pred, zero_division=0),
        recall_score(y_test, pca_pred, zero_division=0),
        recall_score(y_test, hybrid_pred, zero_division=0)
    ]
}

results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("📊 PHASE 5: COMPARATIVE EVALUATION RESULTS")
print("="*80)
print(results_df.round(4))

results_df.to_excel("Phase5_Comparative_Results.xlsx", index=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=results_df.melt(id_vars='Model', value_vars=['Accuracy', 'F1_Score', 'Precision', 'Recall']),
            x='variable', y='value', hue='Model')
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("Phase5_Performance_Comparison.png")
plt.show()

print("\n🔍 Feature Space Analysis (Classical vs Quantum)")

# Compare last timestep features
classical_features = X_test[:, -1, :]
quantum_features = X_test_hybrid

print(f"Classical Feature Dim: {classical_features.shape[1]}")
print(f"Quantum-Enhanced Feature Dim: {quantum_features.shape[1]}")

# PCA Visualization of Feature Spaces
pca_vis = PCA(n_components=2)

classical_2d = pca_vis.fit_transform(classical_features)
quantum_2d = pca_vis.fit_transform(quantum_features)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(classical_2d[:, 0], classical_2d[:, 1], c=y_test, cmap='viridis', alpha=0.7)
plt.title('Classical Feature Space')
plt.xlabel('PC1')
plt.ylabel('PC2')

plt.subplot(1, 2, 2)
plt.scatter(quantum_2d[:, 0], quantum_2d[:, 1], c=y_test, cmap='viridis', alpha=0.7)
plt.title('Quantum-Enhanced Feature Space')
plt.xlabel('PC1')
plt.ylabel('PC2')

plt.tight_layout()
plt.savefig("Phase5_Feature_Space_Comparison.png")
plt.show()

print("\n✅ Phase 5 Completed Successfully!")
print("Files saved:")
print("   • Phase5_Comparative_Results.xlsx")
print("   • Phase5_Performance_Comparison.png")
print("   • Phase5_Feature_Space_Comparison.png")